In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import requests
from bs4 import BeautifulSoup as bs
import re
pd.set_option('display.max_columns', None)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
url = 'https://www.ea.com/games/ea-sports-fc/ratings'
url = 'https://www.ea.com/games/ea-sports-fc/ratings?gender=0&page=2'
abs_url = 'https://www.ea.com'
html = requests.get(url)
soup = bs(html.text)

In [4]:
url = 'https://www.ea.com/games/ea-sports-fc/ratings?gender=0&page=1'
html = requests.get(url)
soup = bs(html.text, 'html.parser')

rows = soup.find_all('tr', class_='Table_row__4INyY')
print(len(rows))  # Si da 0, el selector ya no funciona
print(html.status_code)

print(soup.prettify()[:3000])

0
200
<!DOCTYPE html>
<html data-logged-in="unknown" dir="ltr" lang="en">
 <head>
  <meta charset="utf-8" data-next-head=""/>
  <meta content="width=device-width" data-next-head="" name="viewport"/>
  <link as="font" crossorigin="anonymous" data-next-head="" href="/fonts/fc-26/CruyffSans-Regular.otf" rel="preload" type="font/otf"/>
  <link as="font" crossorigin="anonymous" data-next-head="" href="/fonts/fc-26/CruyffSans-Medium.otf" rel="preload" type="font/otf"/>
  <meta content="https://www.ea.com/meta/ea-sports-fc/fc-26-ratings-meta.jpg" data-next-head="" name="thumbnail"/>
  <title data-next-head="">
   FC 26 Player Ratings Reveal
  </title>
  <meta content="Explore the complete Ratings and PlayStyles for the 17,000+ players available in EA SPORTS FC™ 26." data-next-head="" name="description"/>
  <link crossorigin="anonymous" data-next-head="" href="/_next/static/media/favicon.3156aee2.ico" rel="shortcut icon" type="image/x-icon"/>
  <link data-next-head="" href="https://www.ea.com/

In [3]:
# Pages 1 to 162
players_list = []
for page in range(1,163):
    url = f'https://www.ea.com/games/ea-sports-fc/ratings?gender=0&page={page}'
    abs_url = 'https://www.ea.com'
    html = requests.get(url)
    if html.status_code != 200:
        continue
    soup = bs(html.text)

    for row in soup.findAll('tr', class_= 'Table_row__4INyY'):
        player_info = {}
        for td in row.findAll('td', attrs ={'data-type':'profile'}):
            link = td.find('a', class_='Table_profileCellAnchor__L23hq')
            try:
                player_url = link['href']
            except:
                continue
            # Player rank position
            rank = td.find('div', class_ = 'Table_profileSupLabel__1Kf2t generated_utility19__iBkEh').text.strip('#')
    
            name_html = td.find('div',class_ ='Table_profileContent__Lna_E').text
            # Player name
            player_name = re.sub(r'\#\d*','',name_html)

        for td in row.findAll('td', class_='Table_rowBlock__Ym9Qr'):
            OVR = (td.find('span', class_ = 'Table_statCellValue__0G9QI').text)
            for gen_stat in td.findAll('div', class_='Table_statCell__lGdI4 generated_utility18sm__u9aFt generated_utility17md__x0sP_ Table_groupedStatCell__5Z2vI'):
                gen_stat_name, gen_stat_value = gen_stat.text[0:3], gen_stat.text[3:5]

                player_info.update({
                    'Rank':rank,
                    'Name':player_name,
                    'OVR':OVR,
                    gen_stat_name : gen_stat_value       
                })

            player_html = requests.get(abs_url+player_url)
            player_soup = bs(player_html.text)


           # Player stats
            for ul in player_soup.findAll('ul', class_ ='List_list__iEtjR List_unstyled__9cL_o List_orientationVertical__lt8CQ'):
                for li in ul.findAll('li', class_ = 'List_listItem__u0QBM'):
                    stat_name, stat_value = re.sub('\d+','', li.text), li.text[-2:]
                    if stat_name not in (['Pace', 'Shooting', 'Passing','Defending', 'Physicality', 'Goalkeeping']):
                        player_info[stat_name] = stat_value


            for div in player_soup.findAll('div', class_ = 'ItemGrid_grid__DKaHT ItemGrid_equalRows__oc45E'):
                for attribute in div.children:
                    # Weak foot
                    if attribute.text.startswith('Weak'):
                        player_info['Weak foot'] = attribute.span['aria-label'][0]
                    elif attribute.text.startswith('Posit'):
                        player_info['Position'] = attribute.text[8:].strip()
                    elif attribute.text.startswith('Skill'):
                        player_info['Skill moves'] = attribute.span['aria-label'][0]             
                    elif attribute.text.startswith('Prefe'):
                        player_info['Preferred foot'] = attribute.text[14:].strip()
                    elif attribute.text.startswith('Height'):
                        player_info['Height'] = attribute.text[6:].strip()
                    elif attribute.text.startswith('Weight'):
                        player_info['Weight'] = attribute.text[6:].strip()               
                    elif attribute.text.startswith('Alt'):
                        alt_positions = []
                        for pos in attribute.findAll('div', class_=''):
                            alt_positions.append(pos.text)
                        player_info['Alternative positions'] = ', '.join(alt_positions)

                    elif attribute.text.startswith('Age'):
                        player_info['Age'] = attribute.text[3:].strip()

                    elif attribute.text.startswith('Nation'):
                        player_info['Nation'] = attribute.text[6:].strip()
                    elif attribute.text.startswith('League'):
                        player_info['League'] = attribute.text[6:].strip()

                    elif attribute.text.startswith('Team'):
                        player_info['Team'] = attribute.text[4:].strip()                    
            play_style_ls = []
            
            for style in player_soup.findAll('div', class_ ='AbilitySection_abilityPanelWrapper__biFph'):
                play_style_ls.append(style.h5.text)
            player_info['play style'] = ', '.join(play_style_ls)
            # Player url/link
            player_info['url'] = abs_url+player_url
            players_list.append(player_info)
            
male_players = pd.DataFrame(players_list)

<>:45: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:45: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
/var/folders/j4/ccx0x_pj25n0rb4wn7gxlrwm0000gn/T/ipykernel_86930/2445448077.py:45: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  stat_name, stat_value = re.sub('\d+','', li.text), li.text[-2:]
/var/folders/j4/ccx0x_pj25n0rb4wn7gxlrwm0000gn/T/ipykernel_86930/2445448077.py:11: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  for row in soup.findAll('tr', class_= 'Table_row__4INyY'):


KeyboardInterrupt: 

In [ ]:
# pages 1 to 16
players_list = []
for page in range(1,17):

    url = f'https://www.ea.com/games/ea-sports-fc/ratings?gender=1&page={page}'
    abs_url = 'https://www.ea.com'
    html = requests.get(url)
    if html.status_code != 200:
        continue
    soup = bs(html.text)
    for row in soup.findAll('tr', class_= 'Table_row__4INyY'):
        player_info = {}
        for td in row.findAll('td', attrs ={'data-type':'profile'}):
            link = td.find('a', class_='Table_profileCellAnchor__L23hq')
            try:
                player_url = link['href']
            except:
                continue
            # Player rank position
            rank = td.find('div', class_ = 'Table_profileSupLabel__1Kf2t generated_utility19__iBkEh').text.strip('#')
    
            name_html = td.find('div',class_ ='Table_profileContent__Lna_E').text
            # Player name
            player_name = re.sub(r'\#\d*','',name_html)
        for td in row.findAll('td', class_='Table_rowBlock__Ym9Qr'):
            OVR = (td.find('span', class_ = 'Table_statCellValue__0G9QI').text)
            for gen_stat in td.findAll('div', class_='Table_statCell__lGdI4 generated_utility18sm__u9aFt generated_utility17md__x0sP_ Table_groupedStatCell__5Z2vI'):
                gen_stat_name, gen_stat_value = gen_stat.text[0:3], gen_stat.text[3:5]

                player_info.update({
                    'Rank':rank,
                    'Name':player_name,
                    'OVR':OVR,
                    gen_stat_name : gen_stat_value       
                })

            player_html = requests.get(abs_url+player_url)
            player_soup = bs(player_html.text)


           # Player stats
            for ul in player_soup.findAll('ul', class_ ='List_list__iEtjR List_unstyled__9cL_o List_orientationVertical__lt8CQ'):
                for li in ul.findAll('li', class_ = 'List_listItem__u0QBM'):
                    stat_name, stat_value = re.sub('\d+','', li.text), li.text[-2:]
                    if stat_name not in (['Pace', 'Shooting', 'Passing','Defending', 'Physicality', 'Goalkeeping']):
                        player_info[stat_name] = stat_value


            for div in player_soup.findAll('div', class_ = 'ItemGrid_grid__DKaHT ItemGrid_equalRows__oc45E'):
                for attribute in div.children:
                    # Weak foot
                    if attribute.text.startswith('Weak'):
                        player_info['Weak foot'] = attribute.span['aria-label'][0]
                    elif attribute.text.startswith('Posit'):
                        player_info['Position'] = attribute.text[8:].strip()
                    elif attribute.text.startswith('Skill'):
                        player_info['Skill moves'] = attribute.span['aria-label'][0]             
                    elif attribute.text.startswith('Prefe'):
                        player_info['Preferred foot'] = attribute.text[14:].strip()
                    elif attribute.text.startswith('Height'):
                        player_info['Height'] = attribute.text[6:].strip()
                    elif attribute.text.startswith('Weight'):
                        player_info['Weight'] = attribute.text[6:].strip()               
                    elif attribute.text.startswith('Alt'):
                        alt_positions = []
                        for pos in attribute.findAll('div', class_=''):
                            alt_positions.append(pos.text)
                        player_info['Alternative positions'] = ', '.join(alt_positions)

                    elif attribute.text.startswith('Age'):
                        player_info['Age'] = attribute.text[3:].strip()

                    elif attribute.text.startswith('Nation'):
                        player_info['Nation'] = attribute.text[6:].strip()
                    elif attribute.text.startswith('League'):
                        player_info['League'] = attribute.text[6:].strip()

                    elif attribute.text.startswith('Team'):
                        player_info['Team'] = attribute.text[4:].strip()                    
            play_style_ls = []
            
            for style in player_soup.findAll('div', class_ ='AbilitySection_abilityPanelWrapper__biFph'):
                play_style_ls.append(style.h5.text)
            player_info['play style'] = ', '.join(play_style_ls)
            # Player url/link
            player_info['url'] = abs_url+player_url
            players_list.append(player_info)
            
female_players = pd.DataFrame(players_list)

: 

In [ ]:
integer_columns = ['OVR', 'PAC', 'SHO', 'PAS', 'DRI', 'DEF', 'PHY',
       'Acceleration', 'Sprint Speed', 'Positioning', 'Finishing',
       'Shot Power', 'Long Shots', 'Volleys', 'Penalties', 'Vision', 'Crossing', 
        'Free Kick Accuracy', 'Short Passing', 'Long Passing',
       'Curve', 'Dribbling', 'Agility', 'Balance', 'Reactions', 'Ball Control',
       'Composure', 'Interceptions', 'Heading Accuracy', 'Def Awareness',
       'Standing Tackle', 'Sliding Tackle', 'Jumping', 'Stamina', 'Strength',
       'Aggression', 'GK Diving', 'GK Handling', 'GK Kicking',
       'GK Positioning', 'GK Reflexes']

: 

In [ ]:
print(male_players.columns.tolist())

for file in [female_players, male_players]:    
    for col in integer_columns:
        if file[col].dtype == 'int64':
            continue
        elif file[col].dtype =='float64':  
            continue
        else:
            file.loc[:, col] = file[col].apply(lambda x: re.sub('[A-Za-z]','',x))
            file[col] = file[col].astype('int')

: 

In [ ]:
all_players = pd.concat([male_players, female_players], ignore_index = True)

male_players.to_csv('/kaggle/working/male_players.csv')
female_players.to_csv('/kaggle/working/female_players.csv')
all_players.to_csv('/kaggle/working/all_players.csv')

: 